![Piggy bank](piggy_bank.jpg)

Personal loans are a lucrative revenue stream for banks. The typical interest rate of a two-year loan in the United Kingdom is [around 10%](https://www.experian.com/blogs/ask-experian/whats-a-good-interest-rate-for-a-personal-loan/). This might not sound like a lot, but in September 2022 alone UK consumers borrowed [around £1.5 billion](https://www.ukfinance.org.uk/system/files/2022-12/Household%20Finance%20Review%202022%20Q3-%20Final.pdf), which would mean approximately £300 million in interest generated by banks over two years!

You have been asked to work with a bank to clean the data they collected as part of a recent marketing campaign, which aimed to get customers to take out a personal loan. They plan to conduct more marketing campaigns going forward so would like you to ensure it conforms to the specific structure and data types that they specify so that they can then use the cleaned data you provide to set up a PostgreSQL database, which will store this campaign's data and allow data from future campaigns to be easily imported. 

They have supplied you with a csv file called `"bank_marketing.csv"`, which you will need to clean, reformat, and split the data, saving three final csv files. Specifically, the three files should have the names and contents as outlined below:

## `client.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `age` | `integer` | Client's age in years | N/A |
| `job` | `object` | Client's type of job | Change `"."` to `"_"` |
| `marital` | `object` | Client's marital status | N/A |
| `education` | `object` | Client's level of education | Change `"."` to `"_"` and `"unknown"` to `np.NaN` |
| `credit_default` | `bool` | Whether the client's credit is in default | Convert to `boolean` data type:<br> `1` if `"yes"`, otherwise `0` |
| `mortgage` | `bool` | Whether the client has an existing mortgage (housing loan) | Convert to boolean data type:<br> `1` if `"yes"`, otherwise `0` |

<br>

## `campaign.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `number_contacts` | `integer` | Number of contact attempts to the client in the current campaign | N/A |
| `contact_duration` | `integer` | Last contact duration in seconds | N/A |
| `previous_campaign_contacts` | `integer` | Number of contact attempts to the client in the previous campaign | N/A |
| `previous_outcome` | `bool` | Outcome of the previous campaign | Convert to boolean data type:<br> `1` if `"success"`, otherwise `0`. |
| `campaign_outcome` | `bool` | Outcome of the current campaign | Convert to boolean data type:<br> `1` if `"yes"`, otherwise `0`. |
| `last_contact_date` | `datetime` | Last date the client was contacted | Create from a combination of `day`, `month`, and a newly created `year` column (which should have a value of `2022`); <br> **Format =** `"YYYY-MM-DD"` |

<br>

## `economics.csv`

| column | data type | description | cleaning requirements |
|--------|-----------|-------------|-----------------------|
| `client_id` | `integer` | Client ID | N/A |
| `cons_price_idx` | `float` | Consumer price index (monthly indicator) | N/A |
| `euribor_three_months` | `float` | Euro Interbank Offered Rate (euribor) three-month rate (daily indicator) | N/A |

In [8]:
#import packages
import pandas as pd
import numpy as np
import datetime


In [9]:
#load in & inspect the data
df = pd.read_csv("bank_marketing.csv")
df.info()
df.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   client_id                   41188 non-null  int64  
 1   age                         41188 non-null  int64  
 2   job                         41188 non-null  object 
 3   marital                     41188 non-null  object 
 4   education                   41188 non-null  object 
 5   credit_default              41188 non-null  object 
 6   mortgage                    41188 non-null  object 
 7   month                       41188 non-null  object 
 8   day                         41188 non-null  int64  
 9   contact_duration            41188 non-null  int64  
 10  number_contacts             41188 non-null  int64  
 11  previous_campaign_contacts  41188 non-null  int64  
 12  previous_outcome            41188 non-null  object 
 13  cons_price_idx              411

,client_id,age,job,marital,education,credit_default,mortgage,month,day,contact_duration,number_contacts,previous_campaign_contacts,previous_outcome,cons_price_idx,euribor_three_months,campaign_outcome
4828,4828,58,management,single,professional.course,no,no,may,31,233,1,0,nonexistent,93.994,4.858,no
2973,2973,39,admin.,married,high.school,unknown,yes,may,28,232,2,0,nonexistent,93.994,4.859,no
41146,41146,41,admin.,married,university.degree,no,yes,nov,29,252,4,0,nonexistent,94.767,1.040,yes
38803,38803,52,technician,married,professional.course,no,no,nov,3,202,5,0,nonexistent,92.649,0.714,no
40421,40421,77,management,married,unknown,no,yes,aug,31,160,1,6,success,94.027,0.905,yes


**Objectives:** Subset, clean, and reformat the `bank_marketing.csv` dataset to create and store three new files based on the requirements detailed in the notebook.

# Q1
Split and tidy `bank_marketing.csv`, storing as three DataFrames called `client`, `campaign`, and `economics`, each containing the columns outlined in the notebook and formatted to the data types listed.

In [10]:
#CLIENT
client_orig = df[['client_id','age','job','marital','education','credit_default','mortgage']]
client = client_orig.copy()

#need to fix values/types of: 'job', 'education', 'credit_default', 'mortgage'
#1) 'JOB' - need to replace "." with "_", data type: "object"

#investigate the original values of this column. originally, this column had "object" values -- GOOD
#display(client_orig.job.value_counts(dropna=False))

#make desired changes
client['job'] = client.job.str.replace(".", "_")

#investigate adjusted column, including its data type
#display(client.job.value_counts(dropna=False), client.info())


#2) 'EDUCATION' - need to replace "." with "_", & "unknown" to np.NaN

#investigate the original values. originally, this column had "object" values -- GOOD
#display(client_orig.education.value_counts(dropna=False))

#make desired changes
client['education'] = client.education.str.replace(".","_").replace("unknown",np.nan)

#investigate adjusted column
#display(client.education.value_counts(dropna=False), client.info())


#3) 'CREDIT_DEFAULT' - need to convert to "boolean" data types: 1 if "yes", otherwise 0

#investigate the original values. originally, this column had "object" values
#display(client_orig.credit_default.value_counts(dropna=False))

#make desired changes
client['credit_default'] = client.credit_default.map({"yes":1, "unknown":0, "no":0})
client['credit_default'] = client.credit_default.astype('bool')

#investigate adjusted column
#display(client.credit_default.value_counts(dropna=False), client.info())


#4) 'MORTGAGE' - need to convert to "boolean" data types: 1 if "yes", otherwise 0

#investigate the original values. originally, this column had "object" values
#display(client_orig.mortgage.value_counts(dropna=False))

#make desired changes
client['mortgage'] = client.mortgage.map({"yes":1, "unknown":0, "no":0})
client['mortgage'] = client.mortgage.astype('bool')

#investigate adjusted column
#display(client.mortgage.value_counts(dropna=False), client.info())


display(client.info())
display(client.sample(n=5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   client_id       41188 non-null  int64 
 1   age             41188 non-null  int64 
 2   job             41188 non-null  object
 3   marital         41188 non-null  object
 4   education       39457 non-null  object
 5   credit_default  41188 non-null  bool  
 6   mortgage        41188 non-null  bool  
dtypes: bool(2), int64(2), object(3)
memory usage: 1.6+ MB


None

,client_id,age,job,marital,education,credit_default,mortgage
4579,4579,46,blue-collar,married,basic_4y,False,True
17949,17949,37,admin_,single,high_school,False,True
17674,17674,39,admin_,divorced,high_school,False,False
17671,17671,40,admin_,divorced,university_degree,False,False
35807,35807,47,services,divorced,high_school,False,True


In [11]:
#CAMPAIGN
campaign_orig = df[['client_id', 'number_contacts', 'contact_duration', 'previous_campaign_contacts', 'previous_outcome', 
                    'campaign_outcome', 'day','month']]
campaign = campaign_orig.copy()

#need to fix values/types of: 'previous_outcome', 'campaign_outcome', 'last_contact_date'
#1) PREVIOUS_OUTCOME - convert to "boolean" data types: 1 if "success", otherwise 0

#investigate the original values. originally, this column had "object" values
#display(campaign_orig.previous_outcome.value_counts(dropna=False))

#make desired changes
campaign['previous_outcome'] = campaign.previous_outcome.map({"success":1, "nonexistent":0, "failure":0})
campaign['previous_outcome'] = campaign.previous_outcome.astype('bool')

#investigate adjusted column
#display(campaign.previous_outcome.value_counts(dropna=False), campaign.info())


#2) CAMPAIGN_OUTCOME - convert to "boolean" data types: 1 if "yes", otherwise 0

#investigate the original values. originally, this column had "object" values
#display(campaign_orig.campaign_outcome.value_counts(dropna=False))

#make desired changes
campaign['campaign_outcome'] = campaign.campaign_outcome.map({"yes":1, "no":0})
campaign['campaign_outcome'] = campaign.campaign_outcome.astype('bool')

#investigate adjusted column
#display(campaign.campaign_outcome.value_counts(dropna=False), campaign.info())


#3) LAST_CONTACT_DATE - Create from a combination of 'day', 'month', and a newly created 'year' column 
    #(which should have a value of 2022); format as: "YYYY-MM-DD"

#originally, 'month' is formatted as the first 3 letters of the month (e.g. "mar"). need to convert this to numerical form
campaign['new_month'] = campaign.month.apply(lambda x: datetime.datetime.strptime(x, "%b").month)

#create the 'last_contact_date' column
campaign['last_contact_date'] = pd.to_datetime(campaign.apply(lambda x: datetime.datetime(2022, 
                                            x['new_month'], x['day']).strftime("%Y-%m-%d"), axis=1))
#investigate new column
display(campaign.sample(5))


#fix campaign as specified, & review
campaign = campaign[['client_id','number_contacts','contact_duration','previous_campaign_contacts','previous_outcome',
                    'campaign_outcome','last_contact_date']]

campaign.info()

,client_id,number_contacts,contact_duration,previous_campaign_contacts,previous_outcome,campaign_outcome,day,month,new_month,last_contact_date
9474,9474,1,123,0,False,False,6,jun,6,2022-06-06
25593,25593,1,157,0,False,False,30,nov,11,2022-11-30
38324,38324,1,186,1,False,True,31,oct,10,2022-10-31
2706,2706,1,73,0,False,False,19,may,5,2022-05-19
69,69,1,177,0,False,False,4,may,5,2022-05-04


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 7 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   client_id                   41188 non-null  int64         
 1   number_contacts             41188 non-null  int64         
 2   contact_duration            41188 non-null  int64         
 3   previous_campaign_contacts  41188 non-null  int64         
 4   previous_outcome            41188 non-null  bool          
 5   campaign_outcome            41188 non-null  bool          
 6   last_contact_date           41188 non-null  datetime64[ns]
dtypes: bool(2), datetime64[ns](1), int64(4)
memory usage: 1.6 MB


In [12]:
#ECONOMICS

economics_orig = df[['client_id', 'cons_price_idx', 'euribor_three_months']]
economics = economics_orig.copy()
display(economics_orig.info())

#no adjustments necessary :)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   client_id             41188 non-null  int64  
 1   cons_price_idx        41188 non-null  float64
 2   euribor_three_months  41188 non-null  float64
dtypes: float64(2), int64(1)
memory usage: 965.5 KB


None

# Q2
Save the three DataFrames to csv files, without an index, as `client.csv`, `campaign.csv`, and `economics.csv` respectively.

In [13]:
#Q2

client.to_csv("client.csv", index=False)
campaign.to_csv("campaign.csv", index=False)
economics.to_csv("economics.csv", index=False)